In [2]:
import torch

In [3]:
probs1 = [
    [0.1, 0.2, 0.3, 0.4],
    [0.2, 0.3, 0.4, 0.1],
    [0.3, 0.4, 0.1, 0.2],
    [0.4, 0.1, 0.2, 0.3]
]

In [4]:
probs1 = torch.tensor(probs1)
logprobs1 = torch.log(probs1)

In [5]:
# "zero out" the lower triangle
l, _ = logprobs1.shape
mask = torch.tril(torch.ones(l, l))
logprobs1 = logprobs1.masked_fill(mask == 1, float('-inf'))

In [6]:
logprobs1

tensor([[   -inf, -1.6094, -1.2040, -0.9163],
        [   -inf,    -inf, -0.9163, -2.3026],
        [   -inf,    -inf,    -inf, -1.6094],
        [   -inf,    -inf,    -inf,    -inf]])

In [7]:
# at this point, it is no longer a valid probability distribution
# so we need to fix it

In [8]:
probsmatrix = torch.exp(logprobs1)

In [9]:
probsmatrix

tensor([[0.0000, 0.2000, 0.3000, 0.4000],
        [0.0000, 0.0000, 0.4000, 0.1000],
        [0.0000, 0.0000, 0.0000, 0.2000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

In [10]:
remaining = torch.sum(probsmatrix, dim=1)

In [11]:
remaining

tensor([0.9000, 0.5000, 0.2000, 0.0000])

In [12]:
remaining = 1 - remaining

In [13]:
# for each row count the number of non-zero elements
probnonzero = torch.sum(probsmatrix != 0, dim=1)

In [14]:
probnonzero

tensor([3, 2, 1, 0])

In [15]:
remaining = remaining / probnonzero

In [16]:
remaining

tensor([0.0333, 0.2500, 0.8000,    inf])

In [17]:
# at this point remaining might have some strange values such
# as NaN or inf, so we need to fix it by replacing them with 0
remaining = remaining.masked_fill(torch.isnan(remaining) | torch.isinf(remaining), 0)

In [18]:
remaining = remaining.unsqueeze(1)

In [19]:
remaining

tensor([[0.0333],
        [0.2500],
        [0.8000],
        [0.0000]])

In [21]:
probsmatrix = probsmatrix + remaining

In [22]:
probsmatrix

tensor([[0.0333, 0.2333, 0.3333, 0.4333],
        [0.2500, 0.2500, 0.6500, 0.3500],
        [0.8000, 0.8000, 0.8000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

In [23]:
# this is a little bit janky, but we need to remove the lower triangle again
# using the same mask as before, but this time setting the values to 0
probsmatrix = probsmatrix.masked_fill(mask == 1, 0)

In [25]:
probsmatrix

tensor([[0.0000, 0.2333, 0.3333, 0.4333],
        [0.0000, 0.0000, 0.6500, 0.3500],
        [0.0000, 0.0000, 0.0000, 1.0000],
        [0.0000, 0.0000, 0.0000, 0.0000]])

In [26]:
# and now we go back to log space
logprobs1 = torch.log(probsmatrix)

In [27]:
logprobs1

tensor([[   -inf, -1.4553, -1.0986, -0.8362],
        [   -inf,    -inf, -0.4308, -1.0498],
        [   -inf,    -inf,    -inf,  0.0000],
        [   -inf,    -inf,    -inf,    -inf]])

In [45]:
probs2 = [
    [0.5, 0.5, 0.0, 0.0],
    [0.1, 0.1, 0.4, 0.4],
    [0.0, 0.0, 0.5, 0.5],
    [0.4, 0.4, 0.1, 0.1]
]
probs2 = torch.tensor(probs2)

In [46]:
# now doing the same thing for batched data
batched_probs = torch.stack([probs1, probs2])

In [60]:
def fix_probs(probs):
    logprobs = torch.log(probs)
    batch_size, l, _ = logprobs.shape
    mask = torch.tril(torch.ones(l, l))
    logprobs = logprobs.masked_fill(mask == 1, float('-inf'))
    probsmatrix = torch.exp(logprobs)
    remaining = torch.sum(probsmatrix, dim=2)
    remaining = 1 - remaining

    # for the probnonzero, we actually need to count the number of 
    # zeros in the mask woops
    probnonzero = torch.sum(mask == 0, dim=-1)

    # probnonzero = torch.sum(probsmatrix != 0, dim=2)
    print(remaining)
    print(probnonzero)
    print(probsmatrix)
    remaining = remaining / probnonzero
    probsmatrix = probsmatrix + remaining.unsqueeze(2)
    probsmatrix = probsmatrix.masked_fill(mask == 1, 0)
    logprobs = torch.log(probsmatrix)
    return logprobs

In [61]:
fixed = fix_probs(batched_probs)

tensor([[0.1000, 0.5000, 0.8000, 1.0000],
        [0.5000, 0.2000, 0.5000, 1.0000]])
tensor([3, 2, 1, 0])
tensor([[[0.0000, 0.2000, 0.3000, 0.4000],
         [0.0000, 0.0000, 0.4000, 0.1000],
         [0.0000, 0.0000, 0.0000, 0.2000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.5000, 0.0000, 0.0000],
         [0.0000, 0.0000, 0.4000, 0.4000],
         [0.0000, 0.0000, 0.0000, 0.5000],
         [0.0000, 0.0000, 0.0000, 0.0000]]])


In [62]:
torch.exp(fixed)

tensor([[[0.0000, 0.2333, 0.3333, 0.4333],
         [0.0000, 0.0000, 0.6500, 0.3500],
         [0.0000, 0.0000, 0.0000, 1.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]],

        [[0.0000, 0.6667, 0.1667, 0.1667],
         [0.0000, 0.0000, 0.5000, 0.5000],
         [0.0000, 0.0000, 0.0000, 1.0000],
         [0.0000, 0.0000, 0.0000, 0.0000]]])